# 09 | Retention, opportunity gaps and causal tests

**Author: Chanakya**

Design a measurement system rather than inventing cohort outcomes. No survey, experiment or internal event stream exists in this package. Numerical power results are planning calculations.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='09_retention_and_experiments'
shared.ACTIVE_SOURCES=['atp_future_calendar_rendered_v2']

## 1. Opportunity-adjusted journeys
Use the next dated regular-tour window as an example of a return opportunity. A source-dated tournament window is not a confirmed player appearance or usable hour. Keep first-ever, reactivated, new-entitlement and already-covered users separate.

In [ ]:
future=read('atp_future_windows');future=future[future.regular_atp.astype(str).str.lower()=='true'].copy();future['start_date']=pd.to_datetime(future.start_date);future['end_date']=pd.to_datetime(future.end_date)
journeys=[]
for r in future.itertuples():
 nexts=future[future.start_date>r.end_date].sort_values('start_date')
 if len(nexts):
  n=nexts.iloc[0];journeys.append(dict(acquiring_event=r.event,end_date=r.end_date,next_event=n.event,next_start=n.start_date,gap_days=(n.start_date-r.end_date).days,trigger='After successful play, show time and coverage; confirm usable slot',source_id='atp_future_calendar_rendered_v2'))
journeys=pd.DataFrame(journeys);display(table(journeys,'09_next_event_journeys'))
states=pd.DataFrame([
('First-ever payer','First successful ATP payment, no previous FanCode payment','Pay-to-play, next included event, relevant upgrade','Incremental new payer and portfolio contribution'),
('Reactivated payer','Prior payment but no active access','Restore entitlement clarity, chosen next occasion','Incremental reactivation contribution, not new-payer CAC'),
('Existing payer, ATP uncovered','Current other-sport pass does not cover ATP','Cheapest relevant additional coverage','Incremental total portfolio contribution'),
('ATP already covered','Verified active covering pass','Suppress redundant sale, relevant live/replay nudge','Renewal at genuine expiry and contribution')],columns=['state','eligibility','treatment','primary_outcome']);display(table(states,'09_lifecycle_treatments'))
kpis=pd.DataFrame([
('Incremental new-payer CAC','Incremental campaign cost / treatment-induced first-ever payers','30/60/90 days','Undefined if estimated lift <=0'),
('Portfolio contribution per eligible user','All portfolio net receipts minus variable costs / all randomized users','90 days plus renewal followup','Include non-buyers as zero, refunds and displacement'),
('Second-event engagement','Cohort users watching a different event / all mature eligible cohort users','30 days and next available relevant opportunity','Pre-register 5/15/30-minute sensitivity, not universal habit threshold'),
('Repeat payment','Users making a separate subsequent payment / mature paid cohort','30/60/90 days','Exclude initial prepaid validity and separate ATP/other-sport payment'),
('Renewal','Passes renewed / passes reaching genuine expiry plus grace','Term-specific','Do not call day30 active season access a renewal'),
('Playback failure','Paid viewing attempts failing to start / paid viewing attempts','First session and ongoing','Repeated attempts need user and attempt views'),
('Opt-out','Users opting out / messaged users','7/30 days','Include all treatment contacts'),
('Opportunity-adjusted return','Users returning at next relevant available event / eligible users with that opportunity','Next event plus fixed grace','Report users with no opportunity separately')],columns=['kpi','definition','horizon','guardrail']);display(table(kpis,'09_kpi_dictionary'))

## 2. Power calculation for binary acquisition
Two-sided 5% alpha and 80% power, equal randomized arms, independent users. Conversion baselines and lifts are planning scenarios. Clustered allocation needs a design effect. Multiple primary tests require a revised error budget. Powering acquisition does not automatically power contribution or annual renewal.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
power=[]
for p0 in [.005,.01,.02,.05]:
 for relative in [.1,.2,.3,.5]:
  p1=p0*(1+relative);effect=abs(proportion_effectsize(p1,p0));n=int(np.ceil(NormalIndPower().solve_power(effect_size=effect,alpha=.05,power=.8,ratio=1)))
  power.append(dict(baseline=p0,relative_lift=relative,treatment_rate=p1,absolute_lift=p1-p0,users_per_arm=n,total_users=2*n))
power=pd.DataFrame(power);display(table(power,'09_binary_power_grid'))
plt.figure(figsize=(9,4))
for p0,g in power.groupby('baseline'):plt.plot(g.relative_lift*100,g.total_users,marker='o',label=f'Baseline {p0:.1%}')
plt.yscale('log');plt.xlabel('Relative acquisition lift to detect (%)');plt.ylabel('Total randomized users (log scale)');plt.title('A small media budget cannot guarantee a decisive lift test');plt.legend();fig('09_power_requirements','Analytical planning approximation, 80% power, two-sided 5% alpha. Not an experiment result.')
clusters=pd.DataFrame([dict(mean_cluster_size=m,icc=rho,design_effect=1+(m-1)*rho) for m in [50,200,1000] for rho in [.001,.01,.05]]);display(table(clusters,'09_cluster_design_effect'))

## 3. Contribution uncertainty and mature outcomes
Use a minimum detectable change in mean contribution, not a conversion-only proxy. Required sample depends on the unknown variance. A ratio such as iCAC becomes unstable when incremental payers approach zero. Do not calculate a profitable-looking ratio after conditioning on purchasers only.

In [ ]:
from scipy.stats import norm
z=norm.ppf(.975)+norm.ppf(.8)
contrib=pd.DataFrame([dict(sd_per_eligible_user=sd,minimum_detectable_contribution=delta,users_per_arm=int(np.ceil(2*z*z*sd*sd/(delta*delta)))) for sd in [20,50,100,200] for delta in [1,2,5,10]])
display(table(contrib,'09_contribution_power'))
# Maturity example is calendar arithmetic, not simulated retention data.
acq=pd.date_range('2026-06-01','2026-09-12',freq='7D');cut=pd.Timestamp(CFG['cutoff']);maturity=pd.DataFrame({'acquisition_date':acq,'days_observed':(cut-acq).days})
for horizon in [30,60,90]:maturity[f'eligible_{horizon}d']=maturity.days_observed>=horizon
table(maturity,'09_followup_maturity_example')
pilot=pd.DataFrame([
('0','Instrumentation and entitlement','Verify access, define first-ever payer and payment ledger, assign user IDs','Do not start price tests with unresolved access or missing refunds'),
('1','One acquisition treatment','User-level persistent holdout, separate paid/owned eligibility, fixed horizon','Scale only positive incremental portfolio contribution with acceptable uncertainty'),
('2','One offer treatment','Randomize within event and ownership strata, keep media constant','Reject if discount/credit displaces higher-value purchases'),
('3','Retention nudge','After successful play, relevant next occasion versus holdout','Engagement is leading evidence only, renewal needs term maturity'),
('4','Expansion','Increase only where measured marginal contribution and reachable capacity support it','Pause at nonpositive marginal contribution or worsening playback/refund guardrails')],columns=['stage','test','design','stop_or_scale_rule']);display(table(pilot,'09_pilot_design'))
check('09_experiments',{'positive_sample_sizes':bool((power.users_per_arm>0).all()),'larger_lift_requires_fewer_users':all(g.sort_values('relative_lift').users_per_arm.is_monotonic_decreasing for _,g in power.groupby('baseline')),'cluster_design_effect_at_least_one':bool((clusters.design_effect>=1).all()),'mature_90_subset_30':bool((~maturity.eligible_90d|maturity.eligible_30d).all())})
report('09_retention_findings','Sequence the pilot instead of splitting a small budget across many underpowered tests. Fix and instrument pay-to-play first. Primary business outcome is total portfolio contribution per randomized eligible user. Define mature repeat and genuine renewal separately. Opportunity-adjusted return complements calendar-day return, but cannot hide unavailable events or selectively exclude disengaged users.')

## Source references
These IDs resolve to the preserved bodies, URLs and capture timestamps. Derived tables also retain row-level source IDs where applicable. Case inputs refer to the supplied brief, physical PDF pages 9–14. Review source files resolve through the review collection log. Scenario parameters are in analysis_config.

In [ ]:
references=source_table(['atp_future_calendar_rendered_v2'])
display(table(references,'09_source_references'))